# W8A8 — all methods, one notebook

Runs and compares three W8A8 quantization recipes on **Qwen3-4B**, against the BF16 baseline:

1. **Naive W8A8-INT8** — symmetric per-group weights, dynamic INT8 activations (per-tensor & per-token)
2. **W8A8-FP8** — `float8_e4m3fn` per-group weights, dynamic FP8 activations (per-tensor & per-token)
3. **SmoothQuant W8A8-INT8** — per-channel activation smoothing fused into weights, per-token INT8

The FP model is loaded **once**. Each quantized variant is built, evaluated, then freed before the
next so only one extra model copy is resident at a time (avoids OOM). All numbers land in a single
comparison table at the end.

## 1  Imports & shared helpers

In [ ]:
import math
import copy
import gc


import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


print(f"PyTorch {torch.__version__}")
assert hasattr(torch, 'float8_e4m3fn'), "PyTorch >= 2.1 required for float8_e4m3fn"




In [ ]:
def compute_ppl(
    model,
    tokenizer,
    dataset_name: str = "wikitext",
    dataset_config: str = "wikitext-2-raw-v1",
    split: str = "test",
    max_length: int = 2048,
    stride: int = 512,
    device=None,
    verbose: bool = True,
) -> float:
    """Perplexity on wikitext-2 via sliding window."""
    if device is None:
        device = next(model.parameters()).device


    data = load_dataset(dataset_name, dataset_config, split=split)
    text = "\n\n".join(data["text"])
    input_ids = tokenizer(text, return_tensors="pt").input_ids
    total_tokens = input_ids.size(1)


    if verbose:
        print(f"Corpus: {total_tokens:,} tokens | window={max_length} stride={stride}")


    nlls, prev_end, window_idx = [], 0, 0


    for begin in range(0, total_tokens, stride):
        end = min(begin + max_length, total_tokens)
        chunk = input_ids[:, begin:end].to(device)
        target_len = end - prev_end


        with torch.no_grad():
            logits = model(chunk).logits


        shift_logits = logits[:, -target_len:-1, :].contiguous()
        shift_labels = chunk[:, -target_len + 1:].contiguous()


        nll = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction="sum",
        )
        nlls.append(nll.float())


        if verbose and window_idx % 10 == 0:
            running = torch.exp(torch.stack(nlls).sum() / max(1, end - 1))
            print(f"  window {window_idx:4d} | tokens {end:>8,}/{total_tokens:,} | PPL {running:.2f}")


        prev_end = end
        window_idx += 1
        if end == total_tokens:
            break


    ppl = torch.exp(torch.stack(nlls).sum() / (total_tokens - 1))
    return ppl.item()




def model_size_gb(model) -> float:
    total = sum(p.numel() * p.element_size() for p in model.parameters())
    total += sum(b.numel() * b.element_size() for b in model.buffers())
    return total / 1e9




def get_parent(model, name):
    parent = model
    for part in name.split(".")[:-1]:
        parent = getattr(parent, part)
    return parent




def free(*objs):
    """Delete model objects and reclaim GPU memory between methods."""
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()




# Single dict that accumulates every result for the final table.
results = {}  # label -> dict(ppl=..., size=...)

## 2  FP baseline — load once, measure size and PPL

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
GROUP_SIZE = 128


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fp = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="cuda",
)
model_fp.eval()


size_fp = model_size_gb(model_fp)
print(f"FP model: {size_fp:.2f} GB")

In [ ]:
ppl_fp = compute_ppl(model_fp, tokenizer)
results["FP (BF16)"] = dict(ppl=ppl_fp, size=size_fp)
print(f"\nFP baseline PPL: {ppl_fp:.2f}")




## 3  Method 1 — Naive W8A8-INT8

Symmetric per-group INT8 weights (`group_size=128`), dynamic symmetric INT8 activations.
INT8 uses a *uniform* grid on `[-127, 127]`.

```
weight:  scale_g = max(|W_group|) / 127 ; q = clamp(round(W/scale_g), -127, 127)
act:     scale   = max(|X|) / 127       ; q = clamp(round(X/scale),  -127, 127)
```

In [ ]:
def quantize_w8_groupwise_sym(W: torch.Tensor, group_size: int = 128):
    """Symmetric per-group INT8 weight quantization. Returns (q int8, scales fp32)."""
    W = W.float()
    out_features, in_features = W.shape
    n_groups = math.ceil(in_features / group_size)
    padded = n_groups * group_size
    if padded > in_features:
        W = torch.cat([W, W.new_zeros(out_features, padded - in_features)], dim=1)
    W_grouped = W.view(out_features, n_groups, group_size)
    scales = (W_grouped.abs().amax(dim=2) / 127.0).clamp(min=1e-8)
    q = torch.clamp(torch.round(W_grouped / scales.unsqueeze(2)), -127, 127)
    q = q.view(out_features, padded)[:, :in_features].to(torch.int8)
    return q, scales




def quantize_activation_sym(x: torch.Tensor, mode: str = "per_token"):
    """Dynamic symmetric INT8 activation quantization."""
    x_f = x.float()
    if mode == "per_tensor":
        scale = (x_f.abs().max() / 127.0).clamp(min=1e-8)
    elif mode == "per_token":
        scale = (x_f.abs().amax(dim=-1, keepdim=True) / 127.0).clamp(min=1e-8)
    else:
        raise ValueError(f"Unknown mode: {mode!r}.")
    q = torch.clamp(torch.round(x_f / scale), -127, 127).to(torch.int8)
    return q, scale




class W8A8Linear(nn.Module):
    """Naive W8A8-INT8 linear: per-group INT8 weights, dynamic INT8 activations."""


    def __init__(self, linear: nn.Linear, group_size: int = 128, act_quant: str = "per_token"):
        super().__init__()
        assert act_quant in ("per_tensor", "per_token")
        self.group_size   = group_size
        self.act_quant    = act_quant
        self.in_features  = linear.in_features
        self.out_features = linear.out_features
        q, scales = quantize_w8_groupwise_sym(linear.weight.data, group_size)
        self.register_buffer("w_int8",  q)
        self.register_buffer("w_scale", scales)
        self.bias = linear.bias


    def _dequant_weight(self) -> torch.Tensor:
        out, n_in = self.w_int8.shape
        n_groups  = self.w_scale.shape[1]
        w = self.w_int8.float().view(out, n_groups, -1)
        return (w * self.w_scale.unsqueeze(2)).view(out, n_in)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_q, x_scale = quantize_activation_sym(x, mode=self.act_quant)
        w_fp = self._dequant_weight()
        x_fp = x_q.float() * x_scale
        out = F.linear(x_fp, w_fp, self.bias.float() if self.bias is not None else None)
        return out.to(x.dtype)




def replace_linear_with_w8a8(model, group_size=128, act_quant="per_token"):
    model = copy.deepcopy(model)
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear):
            parent = get_parent(model, name)
            attr = name.split(".")[-1]
            setattr(parent, attr, W8A8Linear(module, group_size=group_size, act_quant=act_quant))
    return model




In [ ]:
for mode in ("per_tensor", "per_token"):
    print(f"Building + evaluating W8A8-INT8 {mode} ...")
    m = replace_linear_with_w8a8(model_fp, group_size=GROUP_SIZE, act_quant=mode)
    m.eval()
    ppl = compute_ppl(m, tokenizer, verbose=False)
    results[f"W8A8-INT8 {mode}"] = dict(ppl=ppl, size=model_size_gb(m))
    print(f"  PPL = {ppl:.2f}")
    free(m)

## 4  Method 2 — W8A8-FP8 (`float8_e4m3fn`)

Same 8-bit storage as INT8, but FP8-E4M3 spaces its values *logarithmically* — dense near zero,
sparse near the `±448` max — which fits long-tailed LLM weight distributions better. Rounding
happens implicitly inside the cast to `float8_e4m3fn`.

In [ ]:
FP8_DTYPE = torch.float8_e4m3fn
FP8_MAX   = torch.finfo(FP8_DTYPE).max  # 448.0




def quantize_w8_groupwise_fp8(W: torch.Tensor, group_size: int = 128):
    """Symmetric per-group FP8-E4M3 weight quantization. Returns (q fp8, scales fp32)."""
    W = W.float()
    out_features, in_features = W.shape
    n_groups = math.ceil(in_features / group_size)
    padded = n_groups * group_size
    if padded > in_features:
        W = torch.cat([W, W.new_zeros(out_features, padded - in_features)], dim=1)
    W_grouped = W.view(out_features, n_groups, group_size)
    scales = (W_grouped.abs().amax(dim=2) / FP8_MAX).clamp(min=1e-8)
    W_scaled = (W_grouped / scales.unsqueeze(2)).clamp(-FP8_MAX, FP8_MAX)
    q = W_scaled.view(out_features, padded)[:, :in_features].to(FP8_DTYPE)
    return q, scales




def quantize_activation_fp8(x: torch.Tensor, mode: str = "per_token"):
    """Dynamic FP8-E4M3 activation quantization."""
    x_f = x.float()
    if mode == "per_tensor":
        scale = (x_f.abs().max() / FP8_MAX).clamp(min=1e-8)
    elif mode == "per_token":
        scale = (x_f.abs().amax(dim=-1, keepdim=True) / FP8_MAX).clamp(min=1e-8)
    else:
        raise ValueError(f"Unknown mode: {mode!r}.")
    q = (x_f / scale).clamp(-FP8_MAX, FP8_MAX).to(FP8_DTYPE)
    return q, scale




class W8A8FP8Linear(nn.Module):
    """Naive W8A8-FP8 linear: per-group FP8 weights, dynamic FP8 activations."""


    def __init__(self, linear: nn.Linear, group_size: int = 128, act_quant: str = "per_token"):
        super().__init__()
        assert act_quant in ("per_tensor", "per_token")
        self.group_size   = group_size
        self.act_quant    = act_quant
        self.in_features  = linear.in_features
        self.out_features = linear.out_features
        q, scales = quantize_w8_groupwise_fp8(linear.weight.data, group_size)
        self.register_buffer("w_fp8",   q)
        self.register_buffer("w_scale", scales)
        self.bias = linear.bias


    def _dequant_weight(self) -> torch.Tensor:
        out, n_in = self.w_fp8.shape
        n_groups  = self.w_scale.shape[1]
        w = self.w_fp8.float().view(out, n_groups, -1)
        return (w * self.w_scale.unsqueeze(2)).view(out, n_in)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_q, x_scale = quantize_activation_fp8(x, mode=self.act_quant)
        w_fp = self._dequant_weight()
        x_fp = x_q.float() * x_scale
        out = F.linear(x_fp, w_fp, self.bias.float() if self.bias is not None else None)
        return out.to(x.dtype)




def replace_linear_with_w8a8fp8(model, group_size=128, act_quant="per_token"):
    model = copy.deepcopy(model)
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear):
            parent = get_parent(model, name)
            attr = name.split(".")[-1]
            setattr(parent, attr, W8A8FP8Linear(module, group_size=group_size, act_quant=act_quant))
    return model




In [ ]:
for mode in ("per_tensor", "per_token"):
    print(f"Building + evaluating W8A8-FP8 {mode} ...")
    m = replace_linear_with_w8a8fp8(model_fp, group_size=GROUP_SIZE, act_quant=mode)
    m.eval()
    ppl = compute_ppl(m, tokenizer, verbose=False)
    results[f"W8A8-FP8 {mode}"] = dict(ppl=ppl, size=model_size_gb(m))
    print(f"  PPL = {ppl:.2f}")
    free(m)




## 5  Method 3 — SmoothQuant W8A8-INT8

Calibrate per-channel activation absmax, compute a per-channel smooth scale
`s[c] = act_scale[c]^alpha / w_scale[c]^(1-alpha)`, fuse `s` into the weights offline, and divide
the activation by `s` before INT8 quantization. This migrates activation-outlier difficulty into
the (easier-to-quantize) weights.

Run with **both** per-tensor and per-token INT8 activations. Per-tensor is the direct comparison
point against naive W8A8-INT8 per-tensor: it isolates how much the smoothing alone recovers when
the activation quantizer is the cheap, outlier-sensitive per-tensor scale.

In [ ]:
def collect_act_scales(
    model: nn.Module,
    tokenizer,
    n_samples: int = 512,
    seq_len: int = 512,
    dataset_name: str = "wikitext",
    dataset_config: str = "wikitext-2-raw-v1",
) -> dict:
    """layer name -> (in_features,) fp32 per-channel activation absmax over calibration set."""
    device = next(model.parameters()).device
    act_scales = {}
    hooks = []


    def make_hook(name):
        def hook(module, inp, out):
            x = inp[0].detach().float().reshape(-1, module.in_features)
            channel_max = x.abs().amax(dim=0)
            if name in act_scales:
                act_scales[name] = torch.maximum(act_scales[name], channel_max)
            else:
                act_scales[name] = channel_max
        return hook


    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(make_hook(name)))


    data = load_dataset(dataset_name, dataset_config, split="train")
    text = "\n\n".join(data["text"])
    tokens = tokenizer(text, return_tensors="pt").input_ids[0]
    n_tokens = min(n_samples * seq_len, tokens.size(0))
    tokens = tokens[:n_tokens]


    with torch.no_grad():
        for i in range(0, n_tokens, seq_len):
            chunk = tokens[i : i + seq_len].unsqueeze(0).to(device)
            model(chunk)


    for h in hooks:
        h.remove()


    return {k: v.cpu().float() for k, v in act_scales.items()}




def _compute_smooth_scale(act_scale, weight, alpha: float = 0.5):
    """s[c] = act_scale[c]^alpha / w_scale[c]^(1-alpha)"""
    w_scale = weight.float().abs().amax(dim=0).clamp(min=1e-8)
    act_scale = act_scale.float().clamp(min=1e-8)
    return (act_scale.pow(alpha) / w_scale.pow(1 - alpha)).clamp(min=1e-8)




class SmoothW8A8Linear(nn.Module):
    """W8A8-INT8 linear with SmoothQuant: smooth scale s fused into weights offline."""


    def __init__(self, in_features: int, out_features: int, bias: bool, group_size: int,
                 act_quant: str = "per_token"):
        super().__init__()
        assert act_quant in ("per_tensor", "per_token")
        self.in_features  = in_features
        self.out_features = out_features
        self.group_size   = group_size
        self.act_quant    = act_quant
        n_groups = math.ceil(in_features / group_size)
        self.register_buffer("w_int8",           torch.zeros(out_features, in_features, dtype=torch.int8))
        self.register_buffer("w_scale",          torch.zeros(out_features, n_groups,    dtype=torch.float32))
        self.register_buffer("act_smooth_scale", torch.ones(in_features,                dtype=torch.float32))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None


    @classmethod
    def from_linear(cls, linear, act_scale, group_size=128, alpha=0.5, act_quant="per_token", init_only=False):
        layer = cls(linear.in_features, linear.out_features, linear.bias is not None, group_size,
                    act_quant=act_quant)
        if init_only:
            return layer
        dev = linear.weight.device
        act_scale = act_scale.to(dev)
        s = _compute_smooth_scale(act_scale, linear.weight.data, alpha=alpha)
        W_smooth = linear.weight.data.float() * s.unsqueeze(0)
        q, scales = quantize_w8_groupwise_sym(W_smooth, group_size)
        layer.w_int8.copy_(q)
        layer.w_scale.copy_(scales)
        layer.act_smooth_scale.copy_(s.cpu().float())
        if linear.bias is not None:
            layer.bias = nn.Parameter(linear.bias.data.clone())
        return layer


    def _dequant_weight(self) -> torch.Tensor:
        out, n_in = self.w_int8.shape
        n_groups  = self.w_scale.shape[1]
        w = self.w_int8.float().view(out, n_groups, -1)
        return (w * self.w_scale.unsqueeze(2)).view(out, n_in)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = self.act_smooth_scale.to(x.device)
        x_smooth = x / s
        x_q, x_scale = quantize_activation_sym(x_smooth, mode=self.act_quant)
        x_fp = x_q.float() * x_scale
        w_fp = self._dequant_weight()
        out = F.linear(x_fp, w_fp, self.bias.float() if self.bias is not None else None)
        return out.to(x.dtype)




def replace_linear_with_smooth_w8a8(model, act_scales, group_size=128, alpha=0.5, act_quant="per_token"):
    model = copy.deepcopy(model)
    for name, module in list(model.named_modules()):
        if not isinstance(module, nn.Linear):
            continue
        if name not in act_scales:
            continue
        parent = get_parent(model, name)
        attr   = name.split(".")[-1]
        setattr(
            parent, attr,
            SmoothW8A8Linear.from_linear(
                module, act_scale=act_scales[name], group_size=group_size, alpha=alpha,
                act_quant=act_quant,
            ),
        )
    return model




In [ ]:
print("Collecting per-channel activation scales (calibration pass) ...")
act_scales = collect_act_scales(model_fp, tokenizer, n_samples=512, seq_len=512)
print(f"Collected scales for {len(act_scales)} Linear layers.")


for mode in ("per_tensor", "per_token"):
    print(f"Building + evaluating SmoothQuant W8A8-INT8 {mode} ...")
    m = replace_linear_with_smooth_w8a8(model_fp, act_scales, group_size=GROUP_SIZE, alpha=0.5, act_quant=mode)
    m.eval()
    ppl = compute_ppl(m, tokenizer, verbose=False)
    results[f"SmoothQuant W8A8-INT8 {mode}"] = dict(ppl=ppl, size=model_size_gb(m))
    print(f"  PPL = {ppl:.2f}")
    free(m)

## 6  Final comparison — all methods

In [ ]:
ppl_fp = results["FP (BF16)"]["ppl"]


order = [
    "FP (BF16)",
    "W8A8-INT8 per_tensor",
    "W8A8-FP8 per_tensor",
    "SmoothQuant W8A8-INT8 per_tensor",
    "W8A8-INT8 per_token",
    "W8A8-FP8 per_token",
    "SmoothQuant W8A8-INT8 per_token",
]


print(f"{'Model':<34}  {'PPL':>8}  {'ΔPPL':>7}  {'Size':>9}")
print("-" * 64)
for label in order:
    if label not in results:
        continue
    r = results[label]
    d = "—" if label == "FP (BF16)" else f"{r['ppl'] - ppl_fp:+.2f}"
    print(f"{label:<34}  {r['ppl']:>8.2f}  {d:>7}  {r['size']:>7.2f} GB")

